In [ ]:
# ==========================================
# Five-Fold Stratified Cross-Validation
# MobileNetV2 - BUSI Dataset
# ==========================================

import os
import random
import numpy as np
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import GlobalAveragePooling2D, Dropout, Dense
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.preprocessing import label_binarize
from tensorflow.keras.preprocessing.image import img_to_array, load_img

# -------------------------------
# 1. Reproducibility
# -------------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# -------------------------------
# 2. Configuration
# -------------------------------
DATA_DIR = "BUSI_dataset"
IMG_SIZE = 192
BATCH_SIZE = 16
LR = 5e-5
EPOCHS = 150
N_SPLITS = 5

# -------------------------------
# 3. Load All Images into Memory
# -------------------------------
images = []
labels = []
class_names = sorted(os.listdir(DATA_DIR))

for label_index, class_name in enumerate(class_names):
    class_path = os.path.join(DATA_DIR, class_name)
    for img_file in os.listdir(class_path):
        img_path = os.path.join(class_path, img_file)
        img = load_img(img_path, target_size=(IMG_SIZE, IMG_SIZE))
        img = img_to_array(img)
        images.append(img)
        labels.append(label_index)

images = np.array(images)
labels = np.array(labels)

# Normalize using MobileNetV2 preprocessing
images = tf.keras.applications.mobilenet_v2.preprocess_input(images)

num_classes = len(class_names)

# -------------------------------
# 4. Stratified 5-Fold
# -------------------------------
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

fold_accuracies = []
fold_f1 = []
fold_auc = []

# -------------------------------
# 5. Cross-Validation Loop
# -------------------------------
for fold, (train_idx, val_idx) in enumerate(skf.split(images, labels)):
    print(f"\n========== Fold {fold+1} ==========")

    X_train, X_val = images[train_idx], images[val_idx]
    y_train, y_val = labels[train_idx], labels[val_idx]

    # Build fresh model for each fold
    base_model = MobileNetV2(
        input_shape=(IMG_SIZE, IMG_SIZE, 3),
        include_top=False,
        weights='imagenet'
    )
    base_model.trainable = False

    x = base_model.output
    x = GlobalAveragePooling2D()(x)
    x = Dropout(0.5)(x)
    outputs = Dense(num_classes, activation='softmax')(x)

    model = Model(inputs=base_model.input, outputs=outputs)

    model.compile(
        optimizer=Adam(learning_rate=LR),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    early_stop = EarlyStopping(
        monitor='val_loss',
        patience=15,
        restore_best_weights=True
    )

    model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        callbacks=[early_stop],
        verbose=1
    )

    # Predictions
    y_pred_probs = model.predict(X_val)
    y_pred = np.argmax(y_pred_probs, axis=1)

    # Metrics
    acc = accuracy_score(y_val, y_pred)
    f1 = f1_score(y_val, y_pred, average='macro')

    y_val_bin = label_binarize(y_val, classes=np.arange(num_classes))
    auc = roc_auc_score(y_val_bin, y_pred_probs, average='macro', multi_class='ovr')

    fold_accuracies.append(acc)
    fold_f1.append(f1)
    fold_auc.append(auc)

    print(f"Fold Accuracy: {acc:.4f}")
    print(f"Fold Macro F1: {f1:.4f}")
    print(f"Fold Macro AUC: {auc:.4f}")

# -------------------------------
# 6. Final Cross-Validation Results
# -------------------------------
print("\n========== Cross-Validation Results ==========")
print(f"Mean Accuracy: {np.mean(fold_accuracies):.4f} ± {np.std(fold_accuracies):.4f}")
print(f"Mean Macro F1: {np.mean(fold_f1):.4f} ± {np.std(fold_f1):.4f}")
print(f"Mean Macro AUC: {np.mean(fold_auc):.4f} ± {np.std(fold_auc):.4f}")
